# Transfer Learning with PyTorch

## Install Packages

In [1]:
import torch
import torch.nn as nn
import torch.optim as opt

In [2]:
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
import torchinfo as ti

In [4]:
import  pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [5]:
from pathlib import Path
import os

In [6]:
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch torchvision version: {torchvision.__version__}")

PyTorch version: 2.12.0
PyTorch torchvision version: 0.27.0


In [7]:
device = "cpu"

## Get the data

In [8]:
data_dir = Path(".") / "data/pizza_steak_suchi"
train_dir = data_dir / "train"
test_dir = data_dir / "test"

we are going to use model from `torchvision.models` so we need to adapt the size of our tensors to the size of the model we are going to use.

Here we are going to use: [`EfficientNet`](https://arxiv.org/pdf/1905.11946)

### Manunal transformation

In [9]:
train_transform = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.ToTensor(),
    # all pretrained models as per the documentation expects this:
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.244, 0.255])
])

test_transform = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.ToTensor(),
    # all pretrained models as per the documentation expects this:
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.244, 0.255])
])

In [10]:
train_dataset = datasets.ImageFolder(root=train_dir, 
                                     transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, 
                                    transform=test_transform)

In [11]:
train_dataloader = DataLoader(dataset=train_dataset, 
                              batch_size=32, 
                              shuffle=True, 
                              num_workers=os.cpu_count()-4, 
                              pin_memory=True, 
                              prefetch_factor=2)

test_dataloader = DataLoader(dataset=test_dataset, 
                             batch_size=32, 
                             num_workers=os.cpu_count()-4, 
                             pin_memory=True, 
                             prefetch_factor=2)

### Automatic transform for EfficientNet

As of `torchvision`v0.13+ there is now support for automatic data transform creation based on the pretrained model we use

In [12]:
# Get the weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT #Default = best available weights

In [13]:
# Get the transformed used to get the pretrained weights
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [14]:
# Create the dataloaders using automatic transform
from modular.datasetup import create_dataloaders

In [15]:
train_dataloader_auto, test_dataloader_auto, class_names = create_dataloaders(train_dir=train_dir,
                                                                              test_dir=test_dir,
                                                                              transform=auto_transforms,
                                                                              batch_size=32)

Train data:
Dataset ImageFolder
    Number of datapoints: 225
    Root location: data/pizza_steak_suchi/train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
Test data:
Dataset ImageFolder
    Number of datapoints: 75
    Root location: data/pizza_steak_suchi/test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )


In [16]:
class_names

['pizza', 'steak', 'sushi']

## Getting a pre-trained model

there are several ways to get an pre-trained model

1. PyTorch domain libraries
2. Libraries like `timm`(torch image models)
3. Hugging Face
4. Paper with code


To chose a model: Experiment...

three things to consider:

a. Speed - how fast does it run (where does it live and for what application like does it need to make a prediction live or not ?)

b. Size - how big is the model (impact the hardware available)

c. Performance - how well does it work on our problem

In [17]:
# setting up a pretrained model
model_effnet = torchvision.models.efficientnet_b0(weights=weights)

In [18]:
feat_layer = model_effnet.features
avg_pool_layer = model_effnet.avgpool
classifier_layer = model_effnet.classifier

In [19]:
# getting a summary of our model
ti.summary(model_effnet, input_size=(1,3,224,224), 
           col_names=["input_size", "output_size", "num_params", "trainable"],col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 1000]            --                   True
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1280, 7, 7]      --                   True
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   True
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    864                  True
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    64                   True
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1, 16, 112,

In [20]:
# changing the output layer to suit our needs and freeze the feature extraction layers (base layers)
# the principle in PyTorch is to switch off the tracking of the gradients
for param in feat_layer.parameters():
    param.requires_grad = False

In [26]:
model_effnet

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [21]:
#update the classifier head to suit our problem 
model_effnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.LazyLinear(out_features=3, bias=True) # so three output as per our problem
)

In [22]:
model_effnet.to(device)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [23]:
# train the efficientnet model
optimizer = opt.Adam(params=model_effnet.parameters())
criterion = nn.CrossEntropyLoss()

In [24]:
next(iter(train_dataloader))

/Users/maximecollet/Desktop/PyTorch/ZTM_nb/Learning_PyTorch/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


[tensor([[[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
           [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
           [-2.1179, -2.1179, -2.1179,  ..., -2.0837, -2.0837, -2.0837],
           ...,
           [ 0.6734,  0.7419,  0.7933,  ...,  1.6495,  1.5810,  1.5468],
           [ 0.6734,  0.7591,  0.8104,  ...,  1.6153,  1.5810,  1.5639],
           [ 0.6049,  0.6906,  0.7591,  ...,  1.5810,  1.5468,  1.5297]],
 
          [[-1.8689, -1.8689, -1.8689,  ..., -1.8206, -1.8046, -1.8046],
           [-1.8689, -1.8689, -1.8689,  ..., -1.8367, -1.8367, -1.8367],
           [-1.8689, -1.8689, -1.8689,  ..., -1.8528, -1.8528, -1.8528],
           ...,
           [ 0.3009,  0.3652,  0.4294,  ...,  1.1688,  1.1045,  1.0723],
           [ 0.3169,  0.3812,  0.4455,  ...,  1.1527,  1.1045,  1.0884],
           [ 0.2687,  0.3330,  0.3973,  ...,  1.1205,  1.0723,  1.0563]],
 
          [[-1.5922, -1.5922, -1.5922,  ..., -1.5460, -1.5460, -1.5614],
           [-

In [25]:
epochs = 20
losses_train = []
for epoch in range(epochs):
    loss_train_acc = 0
    for X_batch, y_batch in train_dataloader_auto:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        # make a prediction
        train_preds = model_effnet(X_batch)
        # compute the loss
        loss_train = criterion(train_preds, y_batch)

        # zero the gradients
        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step()
        loss_train_acc += loss_train.item()
    losses_train.append(loss_train_acc / len(train_dataloader_auto))

    if epoch % 2 == 0:
        print(f"Epoch: {epoch} | Train loss: {losses_train[epoch]}")

Epoch: 0 | Train loss: 1.1364313662052155
Epoch: 2 | Train loss: 0.8741988390684128
Epoch: 4 | Train loss: 0.761737771332264
Epoch: 6 | Train loss: 0.64570177718997
Epoch: 8 | Train loss: 0.6088240593671799
Epoch: 10 | Train loss: 0.5724348574876785
Epoch: 12 | Train loss: 0.5410117357969284
Epoch: 14 | Train loss: 0.5114053823053837
Epoch: 16 | Train loss: 0.5723440460860729
Epoch: 18 | Train loss: 0.49668624997138977
